# O Ônibus do Ponto 348: render final no Colab

Gera o `mp4` de cada episódio (1080x1920, 60 fps, 60 s, com a mixagem final) numa GPU do Colab.

**Antes:** *Ambiente de execução → Alterar tipo de ambiente de execução → GPU* (T4 serve; L4/A100 do Colab Pro são bem mais rápidas).
Depois execute as células **na ordem**. Se a sessão cair, execute de novo da célula 1 até a 5 e a 7: o render continua dos quadros que faltam.

Os vídeos saem em **Google Drive › ponto348 › out**.

## 1. Confere a GPU

In [ ]:
!nvidia-smi -L

## 2. Google Drive (guarda quadros, vozes e vídeos entre as sessões)

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/ponto348'
os.makedirs(BASE, exist_ok=True)
print('pasta de trabalho:', BASE)

## 3. Baixa o projeto (repositório público)

In [ ]:
%cd /content
!rm -rf ponto348_repo
!git clone -q --branch claude/loving-franklin-rmvp6i --depth 1 https://github.com/nunnsx-pixel/Projectonesuspense ponto348_repo
%cd /content/ponto348_repo
!git log --oneline -1

## 4. Instala o Blender 4.2 LTS (leva ~2 min)

In [ ]:
%%bash
set -e
apt-get -qq update
apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libsm6 libxkbcommon0 libegl1 xz-utils > /dev/null
cd /content
for V in 4.2.23 4.2.18 4.2.12 4.2.9 4.2.5 4.2.3; do
  URL="https://download.blender.org/release/Blender4.2/blender-${V}-linux-x64.tar.xz"
  if wget -q --spider "$URL"; then echo "baixando Blender $V"; wget -q "$URL" -O blender.tar.xz; break; fi
done
tar -xf blender.tar.xz
rm -rf blender && mv blender-*-linux-x64 blender
/content/blender/blender -b --version | head -1

## 5. Vozes (os 11 áudios finais, que ficam fora do git)

Baixe o **vozes-finais.zip** na página das prévias (botão *Salvar vozes*), guarde no app Arquivos e escolha-o quando a célula pedir.
Na primeira vez ele é copiado para o Drive; nas próximas sessões a célula já o encontra lá.

In [ ]:
import os, glob, shutil, zipfile
os.chdir('/content')
from google.colab import files
zip_drive = f'{BASE}/vozes-finais.zip'
if not os.path.exists(zip_drive):
    print('Escolha o arquivo vozes-finais.zip:')
    up = files.upload()
    shutil.copy(next(iter(up)), zip_drive)
with zipfile.ZipFile(zip_drive) as z:
    z.extractall('/content/ponto348_repo')
mp3 = sorted(glob.glob('/content/ponto348_repo/ep0*/audio/*.mp3'))
print(len(mp3), 'arquivos de voz (esperado: 11)')
assert len(mp3) == 11, 'faltam arquivos de voz: confira o vozes-finais.zip'

## 6. Mede a velocidade (6 quadros do Ep. 1)

Mostra os segundos por quadro e a projeção de horas do episódio inteiro. Use para escolher a qualidade da célula 7.
Se aparecer `CPU` em vez de `GPU:CUDA`, a GPU não foi ligada: confira a célula 1.

In [ ]:
WIDTH, SAMPLES = '720', '32'      # 1080 / 96 = qualidade máxima (bem mais lento); 720 / 32 = bom custo-benefício
!cd /content/ponto348_repo && BLENDER=/content/blender/blender GPU_BACKEND=cuda BENCH=1 EPS=ep01 WIDTH={WIDTH} SAMPLES={SAMPLES} RENDERS_DIR=/content/bench ./render_tudo.sh

## 7. Renderiza e monta o mp4 de um episódio

Troque `EP` para `ep01`, `ep02` e `ep03` (um por vez). Os quadros ficam no Drive (`ponto348/renders`), então dá para retomar se a sessão cair;
no fim cada episódio vira `ponto348/out/epXX.mp4`. Espaço: ~3–4 GB de quadros por episódio a 720p (~9 GB a 1080p); apague
`ponto348/renders/epXX` do Drive e esvazie a lixeira depois de baixar o vídeo.

In [ ]:
EP = 'ep01'                        # troque: ep01, ep02, ep03
WIDTH, SAMPLES = '720', '32'       # igual à célula 6
!cd /content/ponto348_repo && EPS={EP} WIDTH={WIDTH} SAMPLES={SAMPLES} BLENDER=/content/blender/blender GPU_BACKEND=cuda RENDERS_DIR={BASE}/renders OUT_DIR={BASE}/out ./render_tudo.sh

## 8. Confere o vídeo pronto

Mostra largura, altura, fps, quadros e duração. Depois abra o app **Google Drive** no iPhone › `ponto348` › `out` e salve o `.mp4`.

In [ ]:
%%bash
for f in /content/drive/MyDrive/ponto348/out/*.mp4; do
  echo "$f"
  ffprobe -v error -count_frames -select_streams v:0 -show_entries stream=width,height,r_frame_rate,nb_read_frames,duration -of csv=p=0 "$f"
done
